# Statistical Analysis of Player Behavior and Motivation in Gameplay Telemetry

Udacity AI Masters Capstone - Project 2: Statistical Analysis

**Student:** Robert Mayfield

In [ ]:
import zipfile
import pandas as pd
import numpy as np
from pathlib import Path

DATA_ZIP = Path('../data/raw/data.zip')

## 1. Project Overview

This project uses gameplay telemetry and player survey data from a longitudinal study conducted within PowerWash Simulator to examine whether intrinsic motivation is associated with session duration.

**Research question:** Do players who report higher enjoyment tend to play in longer sessions?

**Null hypothesis:** Session duration does not differ between high enjoyment and low enjoyment player groups.

**Alternative hypothesis:** Session duration differs between high enjoyment and low enjoyment player groups.

Player enjoyment is measured using survey prompts triggered at regular intervals during gameplay. Each prompt asks the player to rate their enjoyment on a Visual Analogue Scale from 0 to 1000 (Vuorre et al., 2023). Players are assigned to high or low enjoyment groups in Section 3 using a median split of each player's mean enjoyment score across all their recorded responses.

Session duration is measured in minutes using the `CurrentSessionLength` variable recorded at each game exit event. This variable is an integer representing total minutes since the game was launched, rounding down, and resetting to zero when the game is closed.

Dataset loading and initial inspection follow the Initial Data Analysis (IDA) framework, which emphasizes systematic examination of data structure, variable types, and quality before hypothesis testing begins (Lusa et al., 2024).

This project contributes to the AI Game Director Studio by establishing a statistical foundation for how gameplay telemetry signals can support reasoning about player engagement and motivation. It does not build the final integrated system but provides the player telemetry reasoning layer for Project 7.

## 2. Dataset Loading and Inspection

The dataset is distributed as a zip archive containing 18 CSV files. This section loads and inspects the five files relevant to this analysis. Three files are excluded: `game_saved.csv` (deprecated in dataset v1.1.0), `subtask_completed.csv` (3.7 GB uncompressed), and `update_current_state.csv` (3.0 GB uncompressed) provide event granularity below the session level that is not needed here.

| File | Contents |
|---|---|
| `demographics.csv` | One row per participant: age, gender, country, login count |
| `study_prompt_answered.csv` | All in-game survey responses across six prompt types |
| `exited_game.csv` | Session exit events including `CurrentSessionLength` |
| `player_logged_in.csv` | Login events per player |
| `job_completed.csv` | Job completion events per player |

In [ ]:
with zipfile.ZipFile(DATA_ZIP) as z:
    with z.open('data/demographics.csv') as f:
        demographics = pd.read_csv(f)

print(f'Shape: {demographics.shape}')
print(f'\nDtypes:\n{demographics.dtypes}')
demographics.head()

In [ ]:
print('Missing values:')
print(demographics.isnull().sum())
print(f'\nAge stats:\n{demographics["age"].describe()}')
print(f'\nGender distribution:\n{demographics["gender"].value_counts()}')
print(f'\nTop 10 countries:\n{demographics["country"].value_counts().head(10)}')

The demographics file contains one row per participant. The dataset includes 11,080 players from 39 countries. Age skews toward younger adults, consistent with the broader Steam gaming population. Missing values in demographic fields reflect participants who declined to provide that information and will be noted during cleaning in Section 3.

In [ ]:
# Loading ~128 MB compressed file. This may take 30 to 60 seconds.
with zipfile.ZipFile(DATA_ZIP) as z:
    with z.open('data/study_prompt_answered.csv') as f:
        survey_raw = pd.read_csv(f)

print(f'Shape: {survey_raw.shape}')
print(f'Unique players: {survey_raw["pid"].nunique():,}')
print(f'\nResponses by prompt type:')
print(survey_raw['LastStudyPromptType'].value_counts())
survey_raw[['pid', 'Time', 'LastStudyPromptType', 'response', 'CurrentSessionLength']].head()

The survey file covers all six Self-Determination Theory constructs used in the study. Enjoyment and Wellbeing (pop-up) are the most frequently triggered prompt types. Autonomy, Competence, and Immersion have fewer responses because those prompts were added partway through the data collection period (Vuorre et al., 2023). This analysis uses Enjoyment responses only. Filtering and aggregation are performed in Section 3.

In [ ]:
with zipfile.ZipFile(DATA_ZIP) as z:
    with z.open('data/exited_game.csv') as f:
        sessions = pd.read_csv(f)

print(f'Shape: {sessions.shape}')
print(f'Unique players: {sessions["pid"].nunique():,}')
print(f'\nCurrentSessionLength stats (minutes):')
print(sessions['CurrentSessionLength'].describe())
sessions[['pid', 'Time', 'CurrentSessionLength', 'CampaignProgressionAmount', 'CurrentGameMode']].head()

Each row in the sessions file represents one game exit event. `CurrentSessionLength` records total minutes since the game was launched, rounded down to the nearest integer. Several data quality issues are visible in the summary statistics: a small number of negative values (likely clock or time zone artifacts), zero-length sessions where the game was launched and exited immediately, and a maximum exceeding 7,000 minutes (over 4 days) that almost certainly represents a session left running rather than active play. These cases will be examined and addressed in Section 3. The distribution is right skewed, with a median around 54 minutes and a long tail of extended sessions. Per-player median session length will be used in the hypothesis test to reduce the influence of outliers.

In [ ]:
with zipfile.ZipFile(DATA_ZIP) as z:
    with z.open('data/player_logged_in.csv') as f:
        logins = pd.read_csv(f)
    with z.open('data/job_completed.csv') as f:
        jobs = pd.read_csv(f)

print(f'Login events:    {len(logins):,} rows, {logins["pid"].nunique():,} unique players')
print(f'Job completions: {len(jobs):,} rows, {jobs["pid"].nunique():,} unique players')

Login count and job completion count are loaded here for use as supplemental features in Section 8. Login count is also available directly from `demographics.logins`, which will be used as a cross-check during preparation.

## 3. Data Cleaning and Preparation

## 4. Descriptive Statistics

## 5. Visualizations

## 6. Hypothesis Test

## 7. Statistical Interpretation and Limitations

## 8. Supplemental Predictive Modeling Comparison

### 8.1 Purpose of Supplemental Section

### 8.2 Target Definition

### 8.3 Train/Test Split

### 8.4 Baseline Model: Logistic Regression

### 8.5 Experimental Model: Random Forest

### 8.6 Model Evaluation

### 8.7 Tradeoffs and Ethical Considerations

## 9. Final Notebook Summary